In [2]:
"""
StructureSurface example: a multi-protein complex (NPM1 pentamer, AlphaFold3 mmCIF).

This demonstrates the structure-aware workflow on a 5-chain assembly read from a
``.cif`` file (mdtraj parses both PDB and mmCIF/PDBx). The NPM1 N-terminal
oligomerization domain forms a pentamer with disordered tails, so it exercises:

1. Per-chain folded/IDR decomposition.
2. Surface classification across all chains.
3. A *cross-chain* contiguous-surface net (the oligomerization interface).
4. FINCHES surface scoring against an IDR, and surface-vs-surface contacts.
5. cis vs trans reach: which surface residues an anchored IDR can touch on its own
chain vs neighbouring chains.

Run with::

python example_multiprotein_npm1.py
"""

import os

import numpy as np

from finches.utils.structure_surface import StructureSurface
from finches.frontend.mpipi_frontend import Mpipi_frontend


CIF = "fold_npm1_pentamer_model_0.cif"



# ------------------------------------------------------------------
# 1. Load the mmCIF complex and decompose every chain.
# ------------------------------------------------------------------
print("Loading", os.path.basename(CIF), "(mmCIF/PDBx)")
ss = StructureSurface(CIF)

chains = sorted({r.chain_index for r in ss.residues})
print(f"\n[1] Decomposition ({len(chains)} chains)")
print(f"    {'chain':>5} {'residues':>9} {'folded':>7} {'idr':>5} {'surface':>8}")
for c in chains:
    recs = [r for r in ss.residues if r.chain_index == c]
    folded = sum(1 for r in recs if r.domain == "folded" and r.modeled)
    idr = sum(1 for r in recs if r.domain == "idr")
    surf = sum(1 for r in recs if r.surface)
    print(f"    {c:>5} {len(recs):>9} {folded:>7} {idr:>5} {surf:>8}")
print(f"    IDR segments across all chains: {len(ss.idr_segments)}")

# ------------------------------------------------------------------
# 2 + 3. Surface + cross-chain net (the pentamer interface shows up as
#        contiguous-surface edges that connect different chains).
# ------------------------------------------------------------------
g = ss.surface_graph
cross = [(a, b) for a, b in g.edges if a[0] != b[0]]
print("\n[2/3] Contiguous-surface net")
print(f"    surface residues : {len(ss.surface_residues)}")
print(f"    net edges        : {g.number_of_edges()}")
print(f"    cross-chain edges: {len(cross)} (oligomerization interface contacts)")

# ------------------------------------------------------------------
# 4. FINCHES scoring. NPM1's surface is highly acidic, so a basic IDR
#    (e.g. an arginine-rich tract) should be broadly attractive.
# ------------------------------------------------------------------
imc = Mpipi_frontend().IMC_object
basic_idr = "RGRGRGRGRGRGRGRGRGRG"
scores = ss.surface_vs_idr(basic_idr, imc)
vals = np.array([v["score"] for v in scores.values()])
print(f"\n[4] Surface vs basic IDR ({basic_idr[:6]}...)")
print(f"    mean score over surface : {vals.mean():+.3f} (negative = attractive)")
print(f"    fraction attractive     : {(vals < 0).mean():.0%}")

# strongest surface-vs-surface contacts (each residue carries its patch context)
svs = ss.surface_vs_surface(imc)
strongest = sorted(svs.items(), key=lambda kv: kv[1])[:6]
print("\n[4b] Strongest surface-vs-surface contacts (incl. cross-chain):")
for (ka, kb), val in strongest:
    a = ss.get_residue(*ka)
    b = ss.get_residue(*kb)
    tag = "cross-chain" if ka[0] != kb[0] else "same-chain"
    print(
        f"      {a.one_letter}{ka[1]}(ch{ka[0]}) <-> "
        f"{b.one_letter}{kb[1]}(ch{kb[0]})  {val:+.3f}  [{tag}]"
    )

# ------------------------------------------------------------------
# 5. cis vs trans reach. Anchor an IDR at one chain's folded junction and
#    see how many reachable surface residues are on the same chain (cis)
#    vs other chains (trans).
# ------------------------------------------------------------------
seg = max(
    (s for s in ss.idr_segments if s["n_anchor"] or s["c_anchor"]),
    key=lambda s: len(s["residues"]),
)
anchor = (seg["c_anchor"] or seg["n_anchor"]).key
idr_len = max(len(seg["residues"]), 40)

reachable = ss.reachable_surface_residues(anchor, idr_len)
cis = sum(1 for k in reachable if k[0] == anchor[0])
trans = len(reachable) - cis
print(f"\n[5] Reach from an IDR (length {idr_len}) anchored at {anchor}")
print(
    f"    reachable surface residues : {len(reachable)} (cis={cis}, trans={trans})"
)



Loading fold_npm1_pentamer_model_0.cif (mmCIF/PDBx)

[1] Decomposition (5 chains)
    chain  residues  folded   idr  surface
        0       294     144   150      101
        1       294     144   150      101
        2       294     144   150      101
        3       294     144   150      101
        4       294     144   150      101
    IDR segments across all chains: 10

[2/3] Contiguous-surface net
    surface residues : 505
    net edges        : 759
    cross-chain edges: 114 (oligomerization interface contacts)

[4] Surface vs basic IDR (RGRGRG...)
    mean score over surface : -0.178 (negative = attractive)
    fraction attractive     : 82%

[4b] Strongest surface-vs-surface contacts (incl. cross-chain):
      K24(ch0) <-> D26(ch0)  -2.673  [same-chain]
      K24(ch0) <-> D36(ch0)  -2.673  [same-chain]
      K24(ch0) <-> D55(ch0)  -2.673  [same-chain]
      K24(ch0) <-> D26(ch1)  -2.673  [cross-chain]
      K24(ch0) <-> D36(ch1)  -2.673  [cross-chain]
      K24(ch0) <-> D55(